# 10.2 Variational Autoencoders

A VAE learns a smooth latent distribution by trading reconstruction accuracy against closeness to a prior.

Autoencoders provide the encoder-decoder skeleton, probability gives Gaussian latents, and KL divergence measures drift from a simple prior. The ELBO makes sampling reliable by pricing latent information.

Save a copy to Drive to edit. This notebook is CPU-only, seeded, and designed to be inspected before you run any cell.

In [ ]:

import math
import random

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.datasets import load_digits
from sklearn.datasets import make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances

SEED = 1007
rng = np.random.default_rng(SEED)
random.seed(SEED)
np.random.seed(SEED)


def sigmoid(a):
    return 1.0 / (1.0 + np.exp(-a))


def stable_log(x):
    return np.log(np.clip(x, 1e-8, 1.0))


def standardize(X):
    X = np.asarray(X, dtype=float)
    center = X.mean(axis=0, keepdims=True)
    scale = X.std(axis=0, keepdims=True)
    scale = np.where(scale < 1e-6, 1.0, scale)
    return (X - center) / scale


def pca_project_reconstruct(X, latent_dim, shrink=1.0):
    X = np.asarray(X, dtype=float)
    latent_dim = int(min(latent_dim, X.shape[0] - 1, X.shape[1]))
    latent_dim = max(latent_dim, 1)
    model = PCA(n_components=latent_dim, random_state=SEED)
    Z = model.fit_transform(X)
    Z = Z * shrink
    X_hat = model.inverse_transform(Z)
    return model, Z, X_hat


def reconstruction_error(X, X_hat):
    return float(0.5 * np.mean(np.sum((X - X_hat) ** 2, axis=1)))


def rbf_two_sample_distance(A, B, gamma=None):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    both = np.vstack([A, B])
    if gamma is None:
        d = pairwise_distances(both[: min(len(both), 80)])
        positive = d[d > 0]
        width = np.median(positive) if positive.size else 1.0
        gamma = 1.0 / (2.0 * width ** 2 + 1e-8)
    Kaa = np.exp(-gamma * pairwise_distances(A, A) ** 2).mean()
    Kbb = np.exp(-gamma * pairwise_distances(B, B) ** 2).mean()
    Kab = np.exp(-gamma * pairwise_distances(A, B) ** 2).mean()
    return float(Kaa + Kbb - 2.0 * Kab)


def sample_latent_decode(model, Z, n_samples, noise_scale=0.75):
    mu = Z.mean(axis=0)
    sigma = Z.std(axis=0) + 1e-6
    Z_new = rng.normal(mu, noise_scale * sigma, size=(n_samples, Z.shape[1]))
    return model.inverse_transform(Z_new)


def make_digit_hard_set(limit=240):
    digits = load_digits()
    X = digits.data.astype(float) / 16.0
    y = digits.target
    X = X[:limit]
    y = y[:limit]
    rows = []
    for i, row in enumerate(X):
        image = row.reshape(8, 8)
        shift = (i % 3) - 1
        moved = np.roll(image, shift=shift, axis=1)
        nuisance = 0.20 * np.sin(np.linspace(0.0, np.pi, 64) + 0.3 * y[i])
        noisy = np.clip(moved.reshape(-1) + nuisance + rng.normal(0.0, 0.06, 64), 0.0, 1.0)
        rows.append(noisy)
    hard = np.asarray(rows)
    interactions = hard[:, :16] * hard[:, 16:32]
    return np.hstack([hard, interactions])


def build_f9_ladder():
    X1 = rng.normal(0.0, 1.0, size=(160, 1))
    X2, y2 = make_moons(n_samples=180, noise=0.06, random_state=SEED)
    means = np.array([[-2.0, 0.0], [1.5, 1.3], [1.2, -1.4]])
    parts = []
    labels = []
    for k, mean in enumerate(means):
        cov = np.array([[0.10 + 0.05 * k, 0.03], [0.03, 0.18]])
        draw = rng.multivariate_normal(mean, cov, size=70)
        parts.append(draw)
        labels.extend([k] * len(draw))
    X3 = np.vstack(parts)
    y3 = np.asarray(labels)
    digits = load_digits()
    X4 = digits.data.astype(float) / 16.0
    y4 = digits.target
    X4 = X4[:240]
    y4 = y4[:240]
    X5 = make_digit_hard_set(limit=240)
    y5 = y4.copy()
    return [
        {"rung": "D1", "name": "1-D Gaussian", "X": standardize(X1), "y": None, "image_shape": None},
        {"rung": "D2", "name": "2-D two moons", "X": standardize(X2), "y": y2, "image_shape": None},
        {"rung": "D3", "name": "3-component mixture", "X": standardize(X3), "y": y3, "image_shape": None},
        {"rung": "D4", "name": "sklearn digits", "X": standardize(X4), "y": y4, "image_shape": (8, 8)},
        {"rung": "D5", "name": "noisy shifted digits with interactions", "X": standardize(X5), "y": y5, "image_shape": (8, 8)},
    ]


def preview_ladder(ladder):
    for item in ladder:
        X = item["X"]
        y = item["y"]
        classes = "none" if y is None else len(np.unique(y))
        print(f"{item['rung']} | {item['name']} | shape={X.shape} | classes={classes}")
        print(np.round(X[:2, : min(6, X.shape[1])], 3))


def first_two_dimensions(X):
    X = np.asarray(X, dtype=float)
    if X.shape[1] == 1:
        return np.column_stack([X[:, 0], np.zeros(len(X))])
    if X.shape[1] == 2:
        return X
    model = PCA(n_components=2, random_state=SEED)
    return model.fit_transform(X)


def panel_axis(ax, X, title, image_shape=None):
    X = np.asarray(X, dtype=float)
    if image_shape is not None and X.shape[1] >= image_shape[0] * image_shape[1]:
        side = image_shape[0] * image_shape[1]
        mosaic = X[:16, :side].reshape(16, image_shape[0], image_shape[1])
        rows = []
        for start in range(0, 16, 4):
            rows.append(np.hstack(mosaic[start:start + 4]))
        ax.imshow(np.vstack(rows), cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
    else:
        xy = first_two_dimensions(X[:120])
        ax.scatter(xy[:, 0], xy[:, 1], s=10, alpha=0.75)
    ax.set_title(title, fontsize=9)


def plot_generated_panels(results, metric_name):
    fig, axes = plt.subplots(2, len(results), figsize=(3.0 * len(results), 5.0))
    for j, result in enumerate(results):
        panel_axis(axes[0, j], result["generated"], result["rung"], result.get("image_shape"))
        axes[1, j].bar([0], [result["metric"]])
        axes[1, j].set_xticks([0])
        axes[1, j].set_xticklabels([metric_name], rotation=30, ha="right")
        axes[1, j].set_title(f"{result['metric']:.3f}", fontsize=9)
    fig.tight_layout()
    plt.show()
    plt.figure(figsize=(6, 3))
    plt.plot([r["rung"] for r in results], [r["metric"] for r in results], marker="o")
    plt.ylabel(metric_name)
    plt.xlabel("dataset complexity")
    plt.title(f"{metric_name} vs complexity")
    plt.grid(True, alpha=0.3)
    plt.show()


## The concept, built once: ELBO and reparameterization

The lesson formula is $z=\mu+\sigma\epsilon$ with $\sigma=\exp(0.5\log\sigma^2)$ and objective $\mathbb{E}_q[\log p_\theta(x\mid z)]-D_{KL}(q\|p)$. We assert the exact lesson values.

In [ ]:
def vae_elbo(mu, logvar, epsilon, recon_nll):
    sigma = float(np.exp(0.5 * logvar))
    z = float(mu + sigma * epsilon)
    kl = float(0.5 * (mu ** 2 + np.exp(logvar) - 1.0 - logvar))
    elbo = float(-recon_nll - kl)
    return sigma, z, kl, elbo

sigma, z, kl, elbo = vae_elbo(0.5, -0.7, 1.2, 0.42)
print(round(sigma, 4), round(z, 3), round(kl, 4), round(elbo, 4))
assert round(sigma, 4) == 0.7047
assert round(z, 3) == 1.346
assert round(kl, 4) == 0.2233
assert round(elbo, 4) == -0.6433

The reusable CPU method uses PCA means as an encoder, a diagonal Gaussian posterior, the reparameterization trick for samples, and an ELBO/NLL proxy that reports both reconstruction and KL.

In [ ]:
def run_topic_model(item):
    X = item["X"]
    latent_dim = 1 if X.shape[1] <= 2 else min(10, X.shape[1] // 4)
    model, mu, X_hat = pca_project_reconstruct(X, latent_dim=latent_dim, shrink=0.95)
    residual = np.mean((X - X_hat) ** 2, axis=1, keepdims=True)
    logvar = np.log(np.clip(residual + 0.05, 0.03, 2.0))
    eps = rng.normal(0.0, 1.0, size=mu.shape)
    Z_sample = mu + np.exp(0.5 * logvar) * eps
    generated = model.inverse_transform(Z_sample[:40])
    recon = reconstruction_error(X, X_hat)
    kl = float(0.5 * np.mean(mu ** 2 + np.exp(logvar) - 1.0 - logvar))
    metric = recon + kl
    return {
        "rung": item["rung"],
        "name": item["name"],
        "metric": metric,
        "kl": kl,
        "generated": generated,
        "image_shape": item["image_shape"],
    }

## The D1-D5 generative ladder

Family F9 uses an inline ladder rather than a shared helper: D1 is a 1-D Gaussian, D2 is two moons, D3 is a mixture, D4 is bundled `sklearn.datasets.load_digits`, and D5 is a harder no-download real/synthetic digits set with shifts, nuisance variation, and feature interactions.

In [ ]:
ladder = build_f9_ladder()
preview_ladder(ladder)

In [ ]:
results = []
for item in ladder:
    result = run_topic_model(item)
    results.append(result)
    print(f"{result['rung']} {result['name']}: elbo_nll_proxy={result['metric']:.4f}, kl={result['kl']:.4f}")

In [ ]:
plot_generated_panels(results, "ELBO / NLL proxy")

## Pitfall on D5: KL collapse to zero

If the decoder ignores $z$, KL can look perfect while generated samples stop using the input. Free bits and warmup keep nonzero latent use.

In [ ]:
d5 = ladder[-1]
X = d5["X"]
model, mu, X_hat = pca_project_reconstruct(X, latent_dim=8, shrink=0.05)
collapsed_kl = float(0.5 * np.mean(mu ** 2))
free_bits = 0.05
warm_model, warm_mu, warm_hat = pca_project_reconstruct(X, latent_dim=8, shrink=0.85)
warm_kl_terms = 0.5 * warm_mu ** 2
warm_kl = float(np.mean(np.maximum(warm_kl_terms, free_bits)))
collapsed_recon = reconstruction_error(X, X_hat)
warm_recon = reconstruction_error(X, warm_hat)
print(f"collapsed kl={collapsed_kl:.4f}, recon={collapsed_recon:.4f}")
print(f"free-bits kl={warm_kl:.4f}, recon={warm_recon:.4f}")

## Evaluate it + practice

- Metric: track ELBO/NLL proxy on every rung and compare against a no-skill baseline that samples noisy training examples.
- Sanity check: generated samples should match the rough support of D1-D3 before you trust D4-D5 panels.
- Ablation: set the KL weight to zero or shrink the encoder means until latent use collapses.
- Failure signal: D5 improves visually while the summary metric worsens, or one mode repeats across many generated panels.
- Reproducibility: keep the seed fixed, then change only one knob at a time.

Practice 1: Change the latent dimension or codebook size and predict which rung changes most.

Practice 2: Replace the D5 nuisance strength with a smaller value and compare the metric curve.

Practice 3: Add one diagnostic printout that would catch the named pitfall before plotting.